# VisClick — Phase 4.4 / D-04: UDA — simplified SHOT (Source Hypothesis Transfer)

**Goal.** Adapt the source-trained YOLOv8s to the desktop domain using **only unlabelled target images** and **no source data at adaptation time**. Method: simplified SHOT (after Liang et al., 2020).

**Corpus.** Same unlabelled desktop target corpus as notebook 13: ScreenSpot desktop slice (~334) + `samples/desktop_seed/` (50) ≈ 384 images. ScreenSpot is reused; the labels are ignored at adaptation time.

**Why simplified.** The full SHOT freezes the classifier head and adapts the feature extractor with information maximization (entropy + diversity) plus self-supervised pseudo-label refinement. For a detection model like YOLOv8 — multi-scale heads, three feature pyramid levels, anchor-free regression — implementing IM at the detector head is not faithful to SHOT's original classification setup.

The simplified version we run is **detection-pseudo-label SHOT**:
1. Run the source-trained detector on all unlabelled target images at high confidence (0.50).
2. The detector's own confident predictions are the only "labels" at adaptation time.
3. Freeze the YOLO head (`freeze=10` does the opposite — freezes backbone; we use the Ultralytics-friendly inverse by unfreezing only the backbone via `freeze=15` to keep head + neck frozen, then doing a short adaptation run).
4. Train the backbone for 15 epochs against the pseudo-labels.
5. Evaluate.

This preserves the SHOT spirit (source-free, head-frozen, target self-supervision) while staying compatible with the Ultralytics trainer.

**Compute reality.** ~5 min for pseudo-labelling, ~20-25 min for 15 epochs at batch 8 imgsz 640. Total ~30 min.

**Honest narrative for the report.** Just like notebook 13, the simplification is named explicitly. The report calls this "detection-pseudo-label SHOT" and reports it as a lower bound on full SHOT.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(
    ["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
import sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics", "pillow", "opencv-python", "matplotlib"],
    check=False,
)
import torch, ultralytics
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("ultralytics:", ultralytics.__version__)


## 14.1 — Bootstrap

Loads `best_source_v8s.pt` and the unlabelled desktop target corpus (ScreenSpot desktop slice + `samples/desktop_seed/`, ~384 images total — same loader as notebook 13).


In [ ]:
import os, glob, shutil, zipfile, tempfile

DRIVE        = "/content/drive/MyDrive/visclick"
SOURCE_WTS   = os.path.join(DRIVE, "weights", "baseline_source", "best_source_v8s.pt")
SHOT_DIR     = os.path.join(DRIVE, "weights", "uda_shot")
REPORTS_TBL  = os.path.join(DRIVE, "reports", "tables")
os.makedirs(SHOT_DIR, exist_ok=True)
os.makedirs(REPORTS_TBL, exist_ok=True)
assert os.path.isfile(SOURCE_WTS), f"Missing source: {SOURCE_WTS}"

# --- Materialise ScreenSpot desktop slice as PNGs (skip if notebook 13 already did it). ---
SCREENSPOT_DIR = "/content/screenspot_desktop_pngs"
os.makedirs(SCREENSPOT_DIR, exist_ok=True)

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False)
from datasets import load_dataset

n_existing = sum(1 for f in os.listdir(SCREENSPOT_DIR) if f.lower().endswith(".png"))
if n_existing < 100:
    HF_CACHE = os.path.join(tempfile.gettempdir(), "visclick_hf_cache")
    ds = load_dataset("rootsautomation/ScreenSpot", split="test", cache_dir=HF_CACHE)
    for i, row in enumerate(ds):
        if row.get("data_source") not in ("macos", "windows"):
            continue
        out = os.path.join(SCREENSPOT_DIR, f"ss_{i:04d}.png")
        if not os.path.isfile(out):
            row["image"].save(out)
print(f"REPORT screenspot_pngs | dir = {SCREENSPOT_DIR} | n = "
      f"{sum(1 for f in os.listdir(SCREENSPOT_DIR) if f.lower().endswith('.png'))}")

TGT_DIRS = [
    SCREENSPOT_DIR,
    "/content/visclick/samples/desktop_seed",
]
TARGET_PATHS = []
for d in TGT_DIRS:
    if not os.path.isdir(d):
        continue
    for f in os.listdir(d):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            TARGET_PATHS.append(os.path.join(d, f))
TARGET_PATHS = sorted(set(TARGET_PATHS))
print(f"REPORT target_corpus | n = {len(TARGET_PATHS)}")
assert len(TARGET_PATHS) >= 100, f"too few unlabelled targets ({len(TARGET_PATHS)})."

CLASSES = ["button", "text", "text_input", "icon", "menu", "checkbox"]


## 14.2 — Source-model pseudo-labelling on the unlabelled target

We use a high confidence (0.50) so the pseudo-labels are reliable. The SHOT paper's intuition is that confident source-model predictions on the target are the safest signal you can extract without any target labels.


In [ ]:
import time
from ultralytics import YOLO

PSEUDO_CONF = 0.50
IMGSZ = 640
WORK = "/content/uda_shot_data"
IMG_DIR = os.path.join(WORK, "images", "train")
LBL_DIR = os.path.join(WORK, "labels", "train")
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(LBL_DIR, exist_ok=True)

teacher = YOLO(SOURCE_WTS)
n_box = 0
t0 = time.time()
for path in TARGET_PATHS:
    results = teacher.predict(path, imgsz=IMGSZ, conf=PSEUDO_CONF, verbose=False)
    r = results[0]
    if len(r.boxes) == 0:
        continue
    stem = os.path.splitext(os.path.basename(path))[0]
    if not os.path.isfile(os.path.join(IMG_DIR, os.path.basename(path))):
        shutil.copy2(path, os.path.join(IMG_DIR, os.path.basename(path)))
    with open(os.path.join(LBL_DIR, stem + ".txt"), "w") as fh:
        for cls, xyxy in zip(r.boxes.cls.tolist(), r.boxes.xyxyn.tolist()):
            x1, y1, x2, y2 = xyxy
            cx = (x1 + x2) / 2; cy = (y1 + y2) / 2
            bw = x2 - x1; bh = y2 - y1
            fh.write(f"{int(cls)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
    n_box += 1
print(f"REPORT shot_pseudo | n = {n_box}/{len(TARGET_PATHS)} | elapsed_s = {time.time()-t0:.1f}")


## 14.3 — Adapt the backbone with head + neck frozen

`freeze=15` keeps YOLOv8s' top 15 modules (head + most of the neck) frozen during training, while letting the backbone adapt. This is the SHOT-spirit setup: source hypothesis (head) is preserved, feature extractor is what changes.

We train for 15 epochs at batch 8, which fits a single Colab session with margin.


In [ ]:
import yaml

EPOCHS = 15

data_yaml = os.path.join(WORK, "data.yaml")
with open(data_yaml, "w") as fh:
    yaml.safe_dump({
        "path": WORK, "train": "images/train", "val": "images/train",
        "names": CLASSES, "nc": len(CLASSES),
    }, fh)

t0 = time.time()
adapted = YOLO(SOURCE_WTS)
adapted.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=8,
    project=SHOT_DIR,
    name="shot_run",
    freeze=15,
    verbose=False,
    plots=False,
)
elapsed = time.time() - t0
ADAPTED_WTS = os.path.join(SHOT_DIR, "shot_run", "weights", "best.pt")
if not os.path.isfile(ADAPTED_WTS):
    ADAPTED_WTS = os.path.join(SHOT_DIR, "shot_run", "weights", "last.pt")
print(f"REPORT shot_train | epochs = {EPOCHS} | elapsed_s = {elapsed:.1f} | weights = {ADAPTED_WTS}")


## 14.4 — Evaluate: CPV on ScreenSpot + hand-corrected mAP

In [ ]:
import subprocess, tempfile, csv

onnx_out = ADAPTED_WTS.replace(".pt", ".onnx")
if not os.path.isfile(onnx_out):
    YOLO(ADAPTED_WTS).export(format="onnx", imgsz=IMGSZ, dynamic=False, opset=12)

with tempfile.TemporaryDirectory() as tmp:
    ss_csv = os.path.join(tmp, "ss.csv")
    subprocess.run([
        sys.executable, "/content/visclick/scripts/run_cpv_screenspot.py",
        "--weights", onnx_out, "--out", ss_csv,
    ], check=True)
    with open(ss_csv) as fh:
        next(fh); ss_overall = next(fh).strip().split(",")
    CPV_SS = float(ss_overall[-1])

    hc_csv = os.path.join(tmp, "hc.csv")
    subprocess.run([
        sys.executable, "/content/visclick/scripts/run_cpv.py",
        "--weights", onnx_out, "--out", hc_csv,
    ], check=True)
    with open(hc_csv) as fh:
        next(fh)
        hc_overall = None
        for line in fh:
            parts = line.strip().split(",")
            if parts and parts[0] == "OVERALL":
                hc_overall = parts; break
    CPV_HC = float(hc_overall[-1]) if hc_overall else float("nan")

print(f"REPORT shot_eval | cpv_screenspot = {CPV_SS:.2f} | cpv_handcorrected = {CPV_HC:.2f}")


## 14.5 — Write `uda_shot.csv`

In [ ]:
OUT_CSV = "/content/visclick/reports/tables/uda_shot.csv"
with open(OUT_CSV, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["method", "epochs", "n_pseudo_imgs", "cpv_screenspot_%", "cpv_handcorrected_%"])
    w.writerow(["shot_simplified", EPOCHS, n_box, f"{CPV_SS:.2f}", f"{CPV_HC:.2f}"])
print(f"REPORT step = WRITE_CSV | path = {OUT_CSV}")
shutil.copy2(OUT_CSV, os.path.join(REPORTS_TBL, "uda_shot.csv"))


## 14.6 — Publish to git

In [ ]:
import os, subprocess

REPO_ROOT = "/content/visclick"
ARTIFACTS = [
    'reports/tables/uda_shot.csv',
]

for rel in ARTIFACTS:
    p = os.path.join(REPO_ROOT, rel)
    assert os.path.exists(p), f"Missing artifact in repo clone: {p}. Run the previous section first."
    print(f"OK  {p}  ({os.path.getsize(p)} bytes)")

token_path = os.path.join(REPO_ROOT, "token")
if not os.path.exists(token_path):
    print(f"WARN: no token file at {token_path}; skipping git push. Copy artifacts in by hand or restore the token.")
else:
    with open(token_path) as fh:
        token = fh.read().strip()

    def run(cmd, **kw):
        r = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True, **kw)
        if r.returncode != 0:
            print("STDOUT:", r.stdout)
            print("STDERR:", r.stderr)
            raise RuntimeError(f"git command failed: {' '.join(cmd)}")
        return r.stdout

    run(["git", "config", "user.email", "hiran@iit.ac.lk"])
    run(["git", "config", "user.name",  "Hiran Abeywardhana"])

    run(["git", "add", *ARTIFACTS])

    status = run(["git", "status", "--porcelain"])
    if not status.strip():
        print("REPORT step = GIT_PUBLISH | status = NOTHING_TO_COMMIT")
    else:
        run(["git", "commit", "-m", 'D-04: simplified SHOT UDA results'])
        url = f"https://{token}@github.com/HiranMadhu/visclick.git"
        push = subprocess.run(["git", "push", url, "HEAD:main"],
                              cwd=REPO_ROOT, capture_output=True, text=True)
        if push.returncode != 0:
            print("PUSH STDERR:", push.stderr)
            raise RuntimeError("git push failed")
        print("REPORT step = GIT_PUBLISH | status = PUSHED")
